In [31]:
import torch 

from IPython import display

from d2l import torch as d2l

batch_size=256

train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)
for X,y in train_iter:
    print(X.shape,y.shape)
    break



torch.Size([256, 1, 28, 28]) torch.Size([256])


In [3]:
num_inputs=784
num_outputs=10

W=torch.normal(0,0.01,size=(num_inputs,num_outputs),requires_grad=True)
b=torch.zeros(num_outputs,requires_grad=True)

In [26]:
X=torch.tensor([[1.0,2.0,3.0],[4.0,5.0,6.0]])
X.sum(0,keepdim=True),X.sum(1,keepdim=True)# 1是 沿着"列"的方向操作的意思

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [ ]:
def softmax(X):
    X_exp=torch.exp(X)
    partition=X_exp.sum(1,keepdims=True)
    print(f"part={partition,X_exp}")
    return X_exp/partition

X=torch.tensor([[0,0,0,0],[1,1,1,1]])
softmax(X)

part=(tensor([[ 4.0000],
        [10.8731]]), tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [2.7183, 2.7183, 2.7183, 2.7183]])),


tensor([[0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500]])

In [6]:
X=torch.normal(0,1,(2,5))
X_prob=softmax(X)
X_prob,X_prob.sum(1)

(tensor([[0.0884, 0.0360, 0.3890, 0.3227, 0.1639],
         [0.0921, 0.1327, 0.1689, 0.1275, 0.4789]]),
 tensor([1., 1.]))

In [ ]:
def net(X):#(batch_size,1,28,28)
    return softmax(torch.matmul(X.reshape(-1,W.shape[0]),W)+b)

y=torch.tensor([0,2])
y_hat=torch.tensor([[0.1,0.3,0.6],[0.3,0.2,0.5]])#y_hat是一个我们已经确定了的值

y_hat[[0,1],y]#x和y上面的范围一一对应的

tensor([0.1000, 0.5000])

In [ ]:
def cross_entropy(y_hat,y):
    return -torch.log(y_hat[range(len(y_hat)),y])

cross_entropy(y_hat,y)

tensor([2.3026, 0.6931])

In [ ]:
def accuracy(y_hat,y): #@save
    if len(y_hat.shape)>1 and y_hat.shape[1]>1:
        y_hat=y_hat.argmax(axis=1)#这一列上的最大值
    cmp=y_hat.type(y.dtype)==y
    return float(cmp.type(y.dtype).sum())

In [14]:
accuracy(y_hat,y)/len(y)

0.5

In [ ]:
class Accumulator: #@save
    
    def __init__(self,n):
        self.data=[0.0]*n
        
    def add(self,*args):
        self.data=[a+float(b) for a,b in zip(self.data,args)]
        
    def reset(self):
        self.data=[0.0]*len(self.data)
        
    def __getitem__(self,idx):
        return self.data[idx]

In [ ]:
def evaluate_accuracy(net,data_iter): #@save
    
    if isinstance(net,torch.nn.Module):
        net.eval()
    metric=d2l.Accumulator(2)
    with torch.no_grad():
        for X,y in data_iter:
            metric.add(accuracy(net(X),y),y.numel())#y.numel是数组的个数
    
    return metric[0]/metric[1]

In [17]:
evaluate_accuracy(net,test_iter)

0.0421

In [28]:
def train_epoch_ch3(net,train_iter,loss,updater):#@save
    
    if isinstance(net,torch.nn.Module):
        net.train()
    
    metric=d2l.Accumulator(3)
    
    for X,y in train_iter:
        y_hat=net(X)
        l=loss(y_hat,y)
            
        if isinstance(updater,torch.optim.Optimizer):
            updater.zero_grad()
            l.mean().backward()
            updater.step()
            
        else:
            l.sum().backward()
            updater(X.shape[0])

        metric.add(float(l.sum()),y.numel())
    return metric[0]/metric[2],metric[1]/metric[2]